In [1]:
!pip install selenium requests pandas Pillow beautifulsoup4 lxml

In [2]:
import time, re, requests, pandas as pd, os
from PIL import Image
from io import BytesIO
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import shutil

In [3]:
options = Options()
options.binary_location = "/usr/bin/chromium"
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1280,900")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_argument("user-agent=Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")
chromedriver_path = shutil.which("chromedriver") or "/usr/lib/chromium/chromedriver"
driver = webdriver.Chrome(service=Service(chromedriver_path), options=options)
print("Driver ready ✅")

Driver ready ✅


In [4]:
driver.get("https://www.houzz.com/photos/kitchen-ideas-and-designs-phbr0-bp~t_709")
WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.CSS_SELECTOR, "img[src]")))
time.sleep(4)
print("Page loaded ✅  Title:", driver.title)

Page loaded ✅  Title: 75 Kitchen Ideas You'll Love - April, 2026 | Houzz


In [5]:
def collect_page(driver, room_type):
    for scroll in range(0, 6000, 600):
        driver.execute_script(f"window.scrollTo(0, {scroll});")
        time.sleep(0.3)
    time.sleep(2)

    soup    = BeautifulSoup(driver.page_source, "lxml")
    results = []
    seen    = set()

    SKIP = ("save","view","add to","sign in","join","explore","browse",
            "see more","load more","similar","get","find","discover",
            "shop","click","learn","read")

    ROOM_KEYS = {"floor","wall","ceiling","kitchen","bedroom","bathroom",
                 "living","dining","office","cabinet","countertop","island",
                 "design","photo","example","style","space","room","layout",
                 "storage","renovation","remodel","transitional","modern",
                 "traditional","contemporary","farmhouse","light","window"}

    for img in soup.find_all("img"):
        src = img.get("src") or img.get("data-src") or ""
        if not src.startswith("http"): continue

        m = re.search(r"-w(\d+)-", src)
        if not m or int(m.group(1)) < 300: continue
        if any(s in src for s in ["logo","icon","avatar","profile"]): continue
        if src in seen: continue

        caption = None
        node = img

        for _ in range(6):
            node = node.parent
            if node is None: break

            for child in node.find_all(["p","span","div","a"], recursive=False):
                text = child.get_text(separator=" ", strip=True)
                words = text.split()

                if len(words) < 10: continue
                if text.lower().startswith(SKIP): continue

                lower_count = sum(1 for w in words if w.islower())
                if lower_count < 5: continue

                if not set(text.lower().split()) & ROOM_KEYS: continue

                caption = text
                break

            if caption: break

        if not caption: continue

        title = (img.get("alt") or "").strip() or " ".join(caption.split()[:6])
        seen.add(src)
        results.append({"title":title,"description":caption,
                        "image_url":src,"room_type":room_type})
    return results

In [6]:
page_data = collect_page(driver, "kitchen")
print(f"Page 1: {len(page_data)} good captions")
for item in page_data[:5]:
    nw = len(item["description"].split())
    print(f"  [{nw} words] {item['description'][:85]}")

Page 1: 39 good captions
  [1068 words] Kitchen Ideas & Designs Beautiful Kitchens by talented designers. No room is quite as
  [34 words] Inspiration for a transitional galley medium tone wood floor and brown floor kitchen 
  [30 words] This kitchen brings out the coziness of the home. With deep green, oak wood, and stun
  [1068 words] Kitchen Ideas & Designs Beautiful Kitchens by talented designers. No room is quite as
  [35 words] Example of a classic l-shaped dark wood floor and brown floor kitchen design in Chica


In [7]:
URLS  = ['https://www.houzz.com/photos/kitchen-ideas-and-designs-phbr0-bp~t_709', 'https://www.houzz.com/photos/modern-kitchen-ideas-phbr1-bp~t_709~s_2206', 'https://www.houzz.com/photos/traditional-kitchen-ideas-phbr1-bp~t_709~s_2107', 'https://www.houzz.com/photos/transitional-kitchen-ideas-phbr1-bp~t_709~s_2209', 'https://www.houzz.com/photos/farmhouse-kitchen-ideas-phbr1-bp~t_709~s_2213', 'https://www.houzz.com/photos/contemporary-kitchen-ideas-phbr1-bp~t_709~s_2207']
PAGES = 400
TARGET = 4000
all_data = []
all_seen = set()
print("Room:", "kitchen", "| Target:", TARGET)

for ui, base_url in enumerate(URLS):
    style = base_url.split("~s_")[-1] if "~s_" in base_url else "main"
    print(f"\n  URL {ui+1}/{len(URLS)} — {style}")
    empty = 0
    for page in range(1, PAGES+1):
        if len(all_data) >= TARGET:
            print("  ✅ Target reached"); break
        driver.get(f"{base_url}?pg={page}")
        print(f"    Page {page:03d}", end=" → ")
        try:
            WebDriverWait(driver, 20).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "img[src]")))
        except:
            print("TIMEOUT"); empty += 1
            if empty >= 3: print("  next URL"); break
            continue
        new = [i for i in collect_page(driver, "kitchen") if i["image_url"] not in all_seen]
        for i in new: all_seen.add(i["image_url"])
        all_data.extend(new)
        print(f"{len(new):3d} new | total: {len(all_data)}")
        empty = 0 if new else empty + 1
        if empty >= 5: print("  No new — next URL"); break
        time.sleep(1.5)
    if len(all_data) >= TARGET: break

print(f"\n✅ Done. Total: {len(all_data)} images")

Room: kitchen | Target: 4000

  URL 1/6 — main
    Page 001 →  40 new | total: 40
    Page 002 →  40 new | total: 80
    Page 003 →  20 new | total: 100
    Page 004 →  32 new | total: 132
    Page 005 →  20 new | total: 152
    Page 006 →  20 new | total: 172
    Page 007 →  20 new | total: 192
    Page 008 →  20 new | total: 212
    Page 009 →  10 new | total: 222
    Page 010 →  30 new | total: 252
    Page 011 →  20 new | total: 272
    Page 012 →  38 new | total: 310
    Page 013 →  20 new | total: 330
    Page 014 →  20 new | total: 350
    Page 015 →  20 new | total: 370
    Page 016 →  20 new | total: 390
    Page 017 →  38 new | total: 428
    Page 018 →  16 new | total: 444
    Page 019 →  34 new | total: 478
    Page 020 →  52 new | total: 530
    Page 021 →  20 new | total: 550
    Page 022 →  20 new | total: 570
    Page 023 →  20 new | total: 590
    Page 024 →  20 new | total: 610
    Page 025 →  26 new | total: 636
    Page 026 →  20 new | total: 656
    Page 027 →  20 

In [8]:
driver.quit()
print('Driver closed ✅')

Driver closed ✅


In [9]:
df = pd.DataFrame(all_data).drop_duplicates(subset=["image_url"]).reset_index(drop=True)
df["wc"] = df["description"].str.split().str.len()
print(f"Total: {len(df)} | Min words: {df['wc'].min()} | Avg: {df['wc'].mean():.1f} | All≥10: {(df['wc']>=10).all()} ✅")
df.head()

Total: 4007 | Min words: 10 | Avg: 254.1 | All≥10: True ✅


,title,description,image_url,room_type,wc
0,Elmhurst,Kitchen Ideas & Designs Beautiful Kitchens by ...,https://st.hzcdn.com/fimgs/56717dbe0cec3211_96...,kitchen,1073
1,Seaside Kitchen Remodel,Inspiration for a transitional galley medium t...,https://st.hzcdn.com/fimgs/81613278085ad1a7_26...,kitchen,34
2,Wedgeway Craftsman Design,This kitchen brings out the coziness of the ho...,https://st.hzcdn.com/fimgs/5ab11d500542c6ea_85...,kitchen,30
3,Parkway Tudor,Kitchen Ideas & Designs Beautiful Kitchens by ...,https://st.hzcdn.com/fimgs/36d1fbb806c893d3_04...,kitchen,1073
4,"Egret Landing | Jupiter, FL",Kitchen Ideas & Designs Beautiful Kitchens by ...,https://st.hzcdn.com/fimgs/6c5135e608b87c30_08...,kitchen,1073


In [10]:
os.makedirs("images/kitchen", exist_ok=True)
H = {"User-Agent":"Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/124.0.0.0 Safari/537.36"}
lp, fail = [], 0
for i, row in df.iterrows():
    fname = "images/kitchen/kitchen_" + str(i).zfill(5) + ".jpg"
    try:
        r = requests.get(row["image_url"], headers=H, timeout=15)
        if r.status_code == 200:
            Image.open(BytesIO(r.content)).convert("RGB").save(fname)
            lp.append(fname)
        else: lp.append(None); fail+=1
    except: lp.append(None); fail+=1
    time.sleep(0.4)
df["local_path"] = lp
print(f"Downloaded: {len(df)-fail} | Failed: {fail}")

Downloaded: 4007 | Failed: 0


In [11]:
df = df.dropna(subset=["local_path"]).drop(columns=["wc"],errors="ignore").reset_index(drop=True)
print(f"Final: {len(df)} images")
df.head()

Final: 4007 images


,title,description,image_url,room_type,local_path
0,Elmhurst,Kitchen Ideas & Designs Beautiful Kitchens by ...,https://st.hzcdn.com/fimgs/56717dbe0cec3211_96...,kitchen,images/kitchen/kitchen_00000.jpg
1,Seaside Kitchen Remodel,Inspiration for a transitional galley medium t...,https://st.hzcdn.com/fimgs/81613278085ad1a7_26...,kitchen,images/kitchen/kitchen_00001.jpg
2,Wedgeway Craftsman Design,This kitchen brings out the coziness of the ho...,https://st.hzcdn.com/fimgs/5ab11d500542c6ea_85...,kitchen,images/kitchen/kitchen_00002.jpg
3,Parkway Tudor,Kitchen Ideas & Designs Beautiful Kitchens by ...,https://st.hzcdn.com/fimgs/36d1fbb806c893d3_04...,kitchen,images/kitchen/kitchen_00003.jpg
4,"Egret Landing | Jupiter, FL",Kitchen Ideas & Designs Beautiful Kitchens by ...,https://st.hzcdn.com/fimgs/6c5135e608b87c30_08...,kitchen,images/kitchen/kitchen_00004.jpg


In [12]:
df.to_csv("kitchen_dataset.csv", index=False)
print("Saved → kitchen_dataset.csv ✅")

Saved → kitchen_dataset.csv ✅


In [13]:
pd.read_csv("kitchen_dataset.csv")

,title,description,image_url,room_type,local_path
0,Elmhurst,Kitchen Ideas & Designs Beautiful Kitchens by ...,https://st.hzcdn.com/fimgs/56717dbe0cec3211_96...,kitchen,images/kitchen/kitchen_00000.jpg
1,Seaside Kitchen Remodel,Inspiration for a transitional galley medium t...,https://st.hzcdn.com/fimgs/81613278085ad1a7_26...,kitchen,images/kitchen/kitchen_00001.jpg
2,Wedgeway Craftsman Design,This kitchen brings out the coziness of the ho...,https://st.hzcdn.com/fimgs/5ab11d500542c6ea_85...,kitchen,images/kitchen/kitchen_00002.jpg
3,Parkway Tudor,Kitchen Ideas & Designs Beautiful Kitchens by ...,https://st.hzcdn.com/fimgs/36d1fbb806c893d3_04...,kitchen,images/kitchen/kitchen_00003.jpg
4,"Egret Landing | Jupiter, FL",Kitchen Ideas & Designs Beautiful Kitchens by ...,https://st.hzcdn.com/fimgs/6c5135e608b87c30_08...,kitchen,images/kitchen/kitchen_00004.jpg
...,...,...,...,...,...
4002,Birmingham Urban Style Kitchen Remodel,The homeowner's favorite part of this kitchen ...,https://st.hzcdn.com/fimgs/4a8159e1060c85a9_78...,kitchen,images/kitchen/kitchen_04002.jpg
4003,Northside Bayview Cape,Beautiful Cape Cod home with commanding views ...,https://st.hzcdn.com/fimgs/8611c03806741cb3_14...,kitchen,images/kitchen/kitchen_04003.jpg
4004,Hamptons Inspired Kitchen,Hamptons-Inspired Kitchen\n \n \nOne cannot he...,https://st.hzcdn.com/fimgs/9d810b890fbbf516_85...,kitchen,images/kitchen/kitchen_04004.jpg
4005,Sacramento Japandi Modern Makeover,Rift-sawn white oak cabinetry was carried all ...,https://st.hzcdn.com/fimgs/85c1a4dc08a8f4dd_98...,kitchen,images/kitchen/kitchen_04005.jpg
